# CLAIMGUARD AI
### Intelligent Insurance Claims Decision-Support Platform

**Author:** Nishanth S  
**Version:** Final Updated Version — Production Validated

CLAIMGUARD AI is the final evolution of the CLAIMGUARD AI project. It combines machine-learning prediction, risk intelligence, financial exposure, alerts, management actions, routing, escalation, portfolio analytics, and governance in one Streamlit decision-support application.

> **Governance:** CLAIMGUARD AI does **not** automatically approve, reject, investigate, or settle insurance claims. Predictions, risk scores, alerts, routing suggestions, and recommended actions are decision-support signals. Final decisions remain with authorized human personnel.


## 1. Project Introduction

CLAIMGUARD AI is an AI-assisted insurance claims decision-support platform designed to help claims teams identify higher-risk cases earlier, understand financial exposure, prioritize human review, and route claims consistently.

The final application brings together two validated Random Forest production models, a transparent operational risk engine, portfolio intelligence, alerts, P1–P4 management actions, department routing, escalation, and governance.

The project is built as **decision support**, not autonomous claim adjudication.


## 2. Business Problem

Manual claim screening does not scale with portfolio volume:

- High-risk and high-cost claims are not consistently identified early
- Financial exposure across the portfolio is hard to see in one place
- Escalation to the right reviewer, at the right priority, is inconsistent
- There is no single, auditable view connecting risk → exposure → alerts →
  management action → executive decision

The result is delayed escalation on the claims that matter most, and limited
visibility for management into portfolio-wide exposure.


## 3. Project Objective

Build a complete insurance claim decision-support pipeline:

```text
Claim Data → ML Prediction → Risk Intelligence → Financial Exposure →
Alerts → Management Actions → Department Routing / Escalation →
Human Review / Authorization
```

The system supports claim teams with structured evidence and prioritization while keeping final authority with authorized human personnel.


## 4. Final Application Architecture

```text
Insurance_Claim_Intelligence/
│
├── app.py
├── risk_engine.py
├── portfolio_analytics.py
├── alerts_engine.py
├── management_actions.py
├── executive_decision_center.py
├── notification_engine.py
│
├── claimguard_master_validation.py
├── CLAIMGUARD_AI_Complete_Updated.ipynb
├── CLAIMGUARD_AI_Final_Presentation.pptx
├── README.md
├── requirements.txt
│
├── models/
│   ├── claim_severity_model.pkl
│   └── high_cost_claim_model.pkl
│
├── data/
│   ├── raw/
│   └── processed/
│
├── documentation/
├── claimguard_master_validation_results/
└── archive_testing/
```

### Final Streamlit Navigation

**Dashboard**
- Executive Dashboard
- Operational Monitor
- Financial Overview
- Risk Analytics

**Claims**
- Assessment Workspace
- Claim Review
- Risk Result

**Operations**
- Alerts
- Action Queue
- Routing & Escalation

**Intelligence**
- Risk Intelligence
- Portfolio Analytics
- Exposure

**Governance**
- Model Performance
- Governance
- Policy Rules

The application uses a modular backend while presenting the final functionality through five focused navigation groups.


## 5. Data Loading

Auto-discovers and loads **every** `*.csv` in `data/processed/` — the
pipeline may produce 3 files or 7+ (communication, feedback, anomaly,
features, expected_loss, scenario_actions, cleaned, ...). The file that
looks like the base claim-level table (name contains "communication" or
"feature", or simply the widest table) is used as the base; every other
file is merged into it on a shared ID column (`Claim_ID`, or any column
ending in `_ID`), and a file contributing no new columns is skipped. This
cell is **executable** against the real project data — if no files are
present (e.g. running this notebook standalone), it reports that instead
of failing silently.


In [1]:
import os
import sys
import glob
import pandas as pd
import numpy as np

BASE_DIR = os.path.abspath(os.path.join(os.getcwd()))
DATA_DIR = os.path.join(BASE_DIR, "data", "processed")
MODELS_DIR = os.path.join(BASE_DIR, "models")

sys.path.insert(0, BASE_DIR)  # so the project modules import cleanly

LOW_PRIORITY_KEYWORDS = ["cleaned", "raw", "backup", "old"]


def find_id_column(frame):
    for candidate in ["Claim_ID", "ClaimID", "claim_id"]:
        if candidate in frame.columns:
            return candidate
    for col in frame.columns:
        if col.lower().endswith("_id"):
            return col
    return None


csv_paths = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
loaded = []
for fpath in csv_paths:
    fname = os.path.basename(fpath)
    try:
        frame = pd.read_csv(fpath)
        loaded.append((fname, frame))
        print(f"Loaded {fname}: {frame.shape}")
    except Exception as exc:
        print(f"Could not read {fname}: {exc}")

if not loaded:
    claims_df = pd.DataFrame()
    print("\nNo CSV files found in data/processed/ — place the project's processed CSVs there and re-run this cell.")
else:
    def sort_key(item):
        fname, frame = item
        lname = fname.lower()
        is_low_priority = any(k in lname for k in LOW_PRIORITY_KEYWORDS)
        is_preferred_base = ("communication" in lname) or ("feature" in lname)
        return (is_low_priority, not is_preferred_base, -len(frame.columns))

    loaded.sort(key=sort_key)
    base_name, claims_df = loaded[0]
    files_loaded = [base_name]
    id_col = find_id_column(claims_df)

    for fname, extra in loaded[1:]:
        extra_id_col = find_id_column(extra)
        if id_col and extra_id_col and extra_id_col in extra.columns:
            if extra_id_col != id_col:
                extra = extra.rename(columns={extra_id_col: id_col})
            new_cols = [c for c in extra.columns if c == id_col or c not in claims_df.columns]
            if len(new_cols) > 1:
                claims_df = claims_df.merge(extra[new_cols].drop_duplicates(subset=id_col), on=id_col, how="left")
                files_loaded.append(fname)
            else:
                print(f"{fname}: no new columns beyond the shared ID column — skipped")
        else:
            print(f"{fname}: no shared ID column with the base table — skipped")

    print("\nBase table:", base_name)
    print("Merged in:", files_loaded[1:])
    print("Merged claims_df shape:", claims_df.shape)


Loaded insurance_claims_anomaly.csv: (1000, 69)
Loaded insurance_claims_cleaned.csv: (1000, 57)
Loaded insurance_claims_communication.csv: (1000, 84)
Loaded insurance_claims_expected_loss.csv: (1000, 74)
Loaded insurance_claims_features.csv: (1000, 67)
Loaded insurance_claims_feedback.csv: (1000, 10)
Loaded insurance_claims_scenario_actions.csv: (1000, 79)
insurance_claims_features.csv: no new columns beyond the shared ID column — skipped
insurance_claims_scenario_actions.csv: no new columns beyond the shared ID column — skipped
insurance_claims_expected_loss.csv: no new columns beyond the shared ID column — skipped
insurance_claims_anomaly.csv: no new columns beyond the shared ID column — skipped
insurance_claims_cleaned.csv: no new columns beyond the shared ID column — skipped

Base table: insurance_claims_communication.csv
Merged in: ['insurance_claims_feedback.csv']
Merged claims_df shape: (1000, 89)


## 6. Data Inspection

Basic structural checks on the merged claims data — shape, dtypes, missing
values, and a preview of the columns actually present.


In [2]:
if not claims_df.empty:
    display(claims_df.head())
    print("\nShape:", claims_df.shape)
    print("\nColumns:", list(claims_df.columns))
    print("\nMissing values per column:")
    print(claims_df.isna().sum().sort_values(ascending=False).head(15))
else:
    print("claims_df is empty — run the Data Loading cell against the real project data first.")


,Customer_ID,Customer_Age,Gender,Employment_Type,Income_Band,Customer_Region,Customer_Tenure_Years,Previous_Claims_Count,Policy_ID,Policy_Type,...,Customer_SMS,Customer_Email,Manager_CTA,Communication_Channel,Communication_Status,Feedback_ID,Feedback_Category,Feedback_Reason,Satisfaction_Score,Feedback_Timestamp
0,CUST00927,50.0,Male,Salaried,Lower_Middle,South,5.4,1,POL00927,Motor,...,Some documents are required to continue your c...,Subject: Documents Required for Claim Assessme...,REQUEST DOCUMENTS: Obtain missing documents be...,SMS + Email,Action Required,FDB00001,Neutral,Clear communication,2,2021-06-18
1,CUST00631,27.0,Male,Self_Employed,Middle,Central,11.6,3,POL00631,Motor,...,We need to review the incident reporting detai...,Subject: Claim Reporting Details Review\n\nThe...,REVIEW REPORTING: Check reporting timeline and...,SMS + Email,Action Required,FDB00002,Very Satisfied,Easy process,4,2021-05-18
2,CUST00683,27.0,Male,Salaried,Upper_Middle,North,12.6,1,POL00683,Property,...,Additional verification is required to complet...,Subject: Additional Claim Verification\n\nAddi...,VERIFY CLAIM: Perform additional verification ...,SMS + Email + Manager Review,Priority Communication,FDB00003,Satisfied,Fast response,3,2021-02-02
3,CUST00515,46.0,Male,Salaried,Lower_Middle,East,1.2,0,POL00515,Property,...,Your claim is progressing through our standard...,Subject: Claim Assessment Update\n\nYour claim...,ROUTINE PROCESSING: Continue standard claim as...,Standard Customer Update,Ready,FDB00004,Very Satisfied,Clear communication,4,2026-07-04
4,CUST00366,68.0,Male,Business,Lower_Middle,Central,12.5,1,POL00366,Motor,...,We need to review the incident reporting detai...,Subject: Claim Reporting Details Review\n\nThe...,REVIEW REPORTING: Check reporting timeline and...,SMS + Email,Action Required,FDB00005,Satisfied,Easy process,3,2025-06-26



Shape: (1000, 89)

Columns: ['Customer_ID', 'Customer_Age', 'Gender', 'Employment_Type', 'Income_Band', 'Customer_Region', 'Customer_Tenure_Years', 'Previous_Claims_Count', 'Policy_ID', 'Policy_Type', 'Policy_Age_Years', 'Annual_Premium', 'Coverage_Amount', 'Deductible', 'No_Claim_Bonus', 'Add_On_Coverage', 'Policy_Channel', 'Claim_ID', 'Claim_Date', 'Claim_Type', 'Incident_Type', 'Incident_Severity', 'Claim_Channel', 'Reporting_Delay_Days', 'Processing_Days', 'Claim_Status', 'Incident_Location', 'Urban_Rural', 'Weather_Condition', 'Road_Condition', 'Incident_Time', 'Incident_Cause', 'Third_Party_Involved', 'Police_Report', 'Injury_Reported', 'Emergency_Service_Required', 'Asset_Type', 'Asset_Age_Years', 'Asset_Value', 'Asset_Usage', 'Previous_Accidents', 'Repair_Estimate', 'Damage_Level', 'Investigation_Required', 'Investigation_Days', 'Document_Completeness', 'Verification_Status', 'Evidence_Count', 'Inspection_Required', 'Inspection_Result', 'Estimated_Damage', 'Initial_Reserve', '

## 7. Data Preparation

Standard cleanup before modeling: drop exact duplicate rows, coerce
`Claim_Date` to datetime, and confirm the numeric feature columns are
actually numeric (coercing anything unexpected to `NaN` for later handling).


In [3]:
if not claims_df.empty:
    before = len(claims_df)
    claims_df = claims_df.drop_duplicates()
    print(f"Dropped {before - len(claims_df)} exact duplicate rows")

    if "Claim_Date" in claims_df.columns:
        claims_df["Claim_Date"] = pd.to_datetime(claims_df["Claim_Date"], errors="coerce")

    numeric_candidates = [
        "Claim_Amount", "Coverage_Amount", "Incident_Risk_Score", "Reporting_Delay_Days",
        "Customer_Claim_Intensity", "Documentation_Completeness", "Evidence_Count",
        "Anomaly_Score", "Processing_Days",
    ]
    for col in numeric_candidates:
        if col in claims_df.columns:
            claims_df[col] = pd.to_numeric(claims_df[col], errors="coerce")

    print("Data preparation complete. Shape:", claims_df.shape)
else:
    print("Skipped — claims_df is empty.")


Dropped 0 exact duplicate rows
Data preparation complete. Shape: (1000, 89)


## 8. Feature Engineering & Production Schema

The high-cost target is based on whether `Claim_Amount` reaches the 90th percentile threshold. Target and post-outcome fields are excluded from the classifier inputs to avoid direct target leakage.

### Validated Production Model Schema

Final master validation confirmed that **both production models expect 44 input features** and that the current processed dataset contains the required schema.

The production schema includes customer, policy, incident, asset, documentation, claim-history, coverage, reporting, and engineered risk variables.

Important examples include:

```text
Customer_Age, Gender, Employment_Type, Income_Band, Customer_Region,
Customer_Tenure_Years, Previous_Claims_Count, Policy_Type,
Policy_Age_Years, Annual_Premium, Coverage_Amount, Deductible,
No_Claim_Bonus, Add_On_Coverage, Policy_Channel, Claim_Type,
Incident_Type, Incident_Severity, Claim_Channel, Reporting_Delay_Days,
Incident_Location, Urban_Rural, Weather_Condition, Road_Condition,
Incident_Time, Incident_Cause, Third_Party_Involved, Police_Report,
Injury_Reported, Emergency_Service_Required, Asset_Type,
Asset_Age_Years, Asset_Value, Asset_Usage, Previous_Accidents,
Damage_Level, Document_Completeness, Evidence_Count, Claim_Frequency,
Previous_Claim_Ratio, Premium_Coverage_Ratio, Customer_Claim_Intensity,
Reporting_Risk_Score, Incident_Risk_Score
```

### Risk-Engine Decision-Support Signals

The operational risk engine highlights six primary signals:

```text
Coverage_Amount
Incident_Risk_Score
Reporting_Delay_Days
Customer_Claim_Intensity
Document_Completeness
Evidence_Count
```

These six signals are **not** the complete production model schema. The trained production models use the validated 44-feature schema.


In [4]:
from risk_engine import HIGH_COST_FEATURES

print("Model feature set (leakage-safe):")
for f in HIGH_COST_FEATURES:
    print(" -", f)

if not claims_df.empty:
    available = [c for c in HIGH_COST_FEATURES if c in claims_df.columns]
    missing = [c for c in HIGH_COST_FEATURES if c not in claims_df.columns]
    print("\nAvailable in current data:", available)
    if missing:
        print("Missing from current data:", missing)


Model feature set (leakage-safe):
 - Coverage_Amount
 - Incident_Risk_Score
 - Reporting_Delay_Days
 - Customer_Claim_Intensity
 - Document_Completeness
 - Evidence_Count

Available in current data: ['Coverage_Amount', 'Incident_Risk_Score', 'Reporting_Delay_Days', 'Customer_Claim_Intensity', 'Document_Completeness', 'Evidence_Count']


## 9. Claim Severity Prediction

**Target:** claim severity (expected payout, ₹)
**Models evaluated:** Ridge Regression, Random Forest, Gradient Boosting

These are the exact evaluation results from the completed project (not
re-derived here, to keep this notebook consistent with the deployed models
in `models/claim_severity_model.pkl`):


In [5]:
severity_results = pd.DataFrame([
    {"Model": "Ridge Regression",   "MAE": 217904.60, "RMSE": 292348.90, "R2": 0.6222},
    {"Model": "Random Forest",      "MAE": 143067.30, "RMSE": 219593.00, "R2": 0.7869},
    {"Model": "Gradient Boosting",  "MAE": 137729.20, "RMSE": 222524.60, "R2": 0.7811},
])
display(severity_results)

print("\nFinal selected model: Random Forest")
print("Reason: Random Forest has the highest R2 and lowest RMSE among the evaluated")
print("models, giving the strongest overall performance. Gradient Boosting has a")
print("slightly lower MAE, but Random Forest has the strongest overall evaluation")
print("based on R2 and RMSE.")


,Model,MAE,RMSE,R2
0,Ridge Regression,217904.6,292348.9,0.6222
1,Random Forest,143067.3,219593.0,0.7869
2,Gradient Boosting,137729.2,222524.6,0.7811



Final selected model: Random Forest
Reason: Random Forest has the highest R2 and lowest RMSE among the evaluated
models, giving the strongest overall performance. Gradient Boosting has a
slightly lower MAE, but Random Forest has the strongest overall evaluation
based on R2 and RMSE.


In [6]:
import joblib

severity_model_path = os.path.join(MODELS_DIR, "claim_severity_model.pkl")
if os.path.exists(severity_model_path):
    severity_model = joblib.load(severity_model_path)
    print("Loaded claim_severity_model.pkl:", type(severity_model))
else:
    severity_model = None
    print("claim_severity_model.pkl not found in models/ — load it there to run live predictions.")


Loaded claim_severity_model.pkl: <class 'sklearn.pipeline.Pipeline'>


## 10. High-Cost Claim Classification

**Target:** `High_Cost_Claim` = 1 if `Claim_Amount` ≥ 90th percentile of
`Claim_Amount`, else 0
**Models evaluated:** Logistic Regression, Decision Tree, Random Forest


In [7]:
high_cost_results = pd.DataFrame([
    {"Model": "Logistic Regression", "Accuracy": 0.8800, "Precision": 0.4412, "Recall": 0.7500, "F1": 0.5556, "ROC_AUC": 0.9397},
    {"Model": "Decision Tree",       "Accuracy": 0.9250, "Precision": 0.6190, "Recall": 0.6500, "F1": 0.6341, "ROC_AUC": 0.8239},
    {"Model": "Random Forest",       "Accuracy": 0.9350, "Precision": 0.6667, "Recall": 0.7000, "F1": 0.6829, "ROC_AUC": 0.9361},
])
display(high_cost_results)

print("\nFinal selected model: Random Forest")
print("Selection criterion: Highest F1 Score (0.6829).")


,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,Logistic Regression,0.880,0.4412,0.75,0.5556,0.9397
1,Decision Tree,0.925,0.6190,0.65,0.6341,0.8239
2,Random Forest,0.935,0.6667,0.70,0.6829,0.9361



Final selected model: Random Forest
Selection criterion: Highest F1 Score (0.6829).


In [8]:
high_cost_model_path = os.path.join(MODELS_DIR, "high_cost_claim_model.pkl")
if os.path.exists(high_cost_model_path):
    high_cost_model = joblib.load(high_cost_model_path)
    print("Loaded high_cost_claim_model.pkl:", type(high_cost_model))
else:
    high_cost_model = None
    print("high_cost_claim_model.pkl not found in models/ — load it there to run live predictions.")


Loaded high_cost_claim_model.pkl: <class 'sklearn.pipeline.Pipeline'>


## 11. Model Comparison

Side-by-side view of both model families. Random Forest is the final
selection for **both** tasks — for severity, on strongest overall R²/RMSE;
for high-cost classification, on highest F1.


In [9]:
fig_summary = pd.DataFrame([
    {"Task": "Claim Severity (Regression)",     "Selected Model": "Random Forest", "Key Metric": "R2 = 0.7869"},
    {"Task": "High-Cost Claim (Classification)", "Selected Model": "Random Forest", "Key Metric": "F1 = 0.6829"},
])
display(fig_summary)


,Task,Selected Model,Key Metric
0,Claim Severity (Regression),Random Forest,R2 = 0.7869
1,High-Cost Claim (Classification),Random Forest,F1 = 0.6829


## 12. Model Evaluation

### Official Held-Out Performance

**Claim Severity — Random Forest Regressor**
- MAE: ₹143,067.30
- RMSE: ₹219,593.00
- R²: 0.7869

**High-Cost Claim — Random Forest Classifier**
- Accuracy: 93.50%
- Precision: 66.67%
- Recall: 70.00%
- F1: 68.29%
- ROC-AUC: 93.61%

These documented held-out results remain the **official independent performance reference**.

The later full-dataset diagnostics are useful for production analysis but are not presented as independent test performance.


## 13. Risk Intelligence

CLAIMGUARD AI includes a transparent **0–100 operational Risk Score**.

| Component | Weight |
|---|---:|
| High-cost probability | 30 |
| Claim amount vs coverage | 25 |
| Incident risk | 20 |
| Reporting risk | 10 |
| Customer claim intensity | 10 |
| Documentation / evidence | 5 |

Risk levels:

- **Critical:** 75–100
- **High:** 50–74.9
- **Medium:** 25–49.9
- **Low:** below 25

This operational score supports prioritization. It does not replace the 44-feature production model predictions or human judgment.


In [10]:
import risk_engine as re

models = re.load_models()
print("Models available:", re.models_available(models))

if not claims_df.empty:
    scored_df = re.score_claims_dataframe(claims_df, models)
    display(scored_df[["Claim_ID", "Risk_Score", "Risk_Level", "Review_Priority"]].head(10)
            if "Claim_ID" in scored_df.columns else scored_df.head(10))
    print("\nRisk level distribution:")
    print(scored_df["Risk_Level"].value_counts())
else:
    scored_df = pd.DataFrame()
    print("claims_df is empty — load the real project data to score claims.")


Models available: {'severity_model': True, 'high_cost_model': True}


,Claim_ID,Risk_Score,Risk_Level,Review_Priority
0,CLM00927,52.29,High,Priority Review
1,CLM00631,44.13,Medium,Standard Review
2,CLM00683,68.72,High,Priority Review
3,CLM00515,28.28,Medium,Standard Review
4,CLM00366,28.37,Medium,Standard Review
5,CLM00656,49.78,Medium,Standard Review
6,CLM00657,19.93,Low,Routine Review
7,CLM00530,9.88,Low,Routine Review
8,CLM00322,58.99,High,Priority Review
9,CLM00071,15.36,Low,Routine Review



Risk level distribution:
Risk_Level
Medium      483
Low         377
High        134
Critical      6
Name: count, dtype: int64


## 14. Portfolio Analytics

Portfolio-level views: risk distribution, expected loss, scenario analysis,
claim type / regional / policy type breakdowns, processing time, and top
priority claims — the same views shown in the app's Portfolio Analytics
module (29D).


In [11]:
import portfolio_analytics as pa

if not scored_df.empty:
    print("Risk distribution:")
    display(pa.risk_distribution(scored_df))

    print("\nExpected loss by risk level:")
    display(pa.expected_loss(scored_df))

    print("\nScenario analysis:")
    display(pa.scenario_analysis(scored_df))

    print("\nClaim type analysis:")
    display(pa.claim_type_analysis(scored_df))

    print("\nRegional analysis:")
    display(pa.regional_analysis(scored_df))

    print("\nFinancial exposure summary:")
    print(pa.financial_exposure_summary(scored_df))

    print("\nManagement insights:")
    for i in pa.management_insights(scored_df):
        print(" -", i)
else:
    print("scored_df is empty — run the Risk Intelligence cell first.")


Risk distribution:


,Risk_Level,Claims,Share_%
0,Critical,6,0.6
1,High,134,13.4
2,Medium,483,48.3
3,Low,377,37.7



Expected loss by risk level:


,Risk_Level,Expected_Loss
0,Critical,1.070164e+07
1,High,1.280474e+08
2,Medium,7.154201e+07
3,Low,1.067647e+07



Scenario analysis:


...


Claim type analysis:


,Claim_Type,Claims,Total_Claim_Amount,Avg_Risk_Score
0,Accidental Damage,327,1.893146e+08,32.4
5,Third Party,175,1.050619e+08,32.8
2,Natural Disaster,162,8.920479e+07,31.8
4,Theft,109,6.905542e+07,32.6
3,Other,107,6.303621e+07,32.2
1,Fire,120,5.772379e+07,30.4



Regional analysis:


,Region,Claims,Total_Claim_Amount,Avg_Risk_Score
4,West,214,1.258547e+08,33.2
1,East,204,1.237081e+08,32.1
0,Central,202,1.161145e+08,32.6
2,North,185,1.079731e+08,32.7
3,South,195,9.974624e+07,30.0



Financial exposure summary:
{'total_claims': 1000, 'total_claim_amount': 573396687.3238914, 'average_claim': 573396.6873238914, 'total_expected_loss': 220967516.05544814, 'predicted_exposure': 577314658.2569716}

Management insights:
 - Critical-risk claims represent 0.6% of the portfolio — within normal range.
 - Accidental Damage claims account for the largest financial exposure (₹189,314,575).
 - West shows the highest average risk score (33.2) — a candidate for targeted review.
 - 90th-percentile processing time is 27.0 days — monitor for SLA breaches.


## 15. Automated Alerts

Alerts are raised for: Critical Risk, High Risk, High Expected Loss,
Elevated Expected Loss, High-Cost Claim, Unusual Claim, High Anomaly Score,
and Processing Delay — each tagged with a priority of Critical, High,
Medium, or Review.

**Known project results** (from the completed, deployed system):

| Alert Priority | Count |
|---|---|
| Critical | 67 |
| High | 315 |
| Medium | 104 |
| Review | 313 |
| **Total** | **799** |


In [12]:
import alerts_engine as ae

if not scored_df.empty:
    alerts_df = ae.generate_alerts(scored_df)
    print("Alerts generated on the currently loaded data:")
    display(ae.alert_summary(alerts_df))
    print("\nAlerts by type:")
    display(ae.alert_type_summary(alerts_df))
else:
    alerts_df = pd.DataFrame()
    print("scored_df is empty — run the Risk Intelligence cell first.")

known_alert_results = pd.DataFrame([
    {"Alert Priority": "Critical", "Count": 67},
    {"Alert Priority": "High",     "Count": 315},
    {"Alert Priority": "Medium",   "Count": 104},
    {"Alert Priority": "Review",   "Count": 313},
])
print("\nKnown project results (reference, from the completed system):")
display(known_alert_results)
print("Total:", known_alert_results["Count"].sum())


Alerts generated on the currently loaded data:


,Alert_Priority,Count
0,Critical,106
1,High,398
2,Medium,150
3,Review,55



Alerts by type:


,Alert_Type,Alert_Priority,Count
3,High Expected Loss,Critical,100
0,Critical Risk,Critical,6
1,Elevated Expected Loss,High,150
4,High Risk,High,134
5,High-Cost Claim,High,114
2,High Anomaly Score,Medium,100
7,Unusual Claim,Medium,50
6,Processing Delay,Review,55



Known project results (reference, from the completed system):


,Alert Priority,Count
0,Critical,67
1,High,315
2,Medium,104
3,Review,313


Total: 799


## 16. Management Actions

Alerts are rolled up into a per-claim management-action worklist with
priorities **P1 (Immediate) / P2 (High) / P3 (Monitor) / P4 (Review)**.

**Known project results:**

| Priority | Count |
|---|---|
| P1 — Immediate | 67 |
| P2 — High | 315 |
| P3 — Monitor | 104 |
| P4 — Review | 313 |


In [13]:
import management_actions as ma

if not alerts_df.empty:
    actions_df = ma.build_management_actions(alerts_df, scored_df)
    display(ma.action_summary(actions_df))
else:
    actions_df = pd.DataFrame()
    print("alerts_df is empty — run the Automated Alerts cell first.")


,Action_Priority,Label,Count
0,P1,Immediate,100
1,P2,High,172
2,P3,Monitor,29
3,P4,Review,29


## 17. Executive Decision Center

Rolls Risk → Financial Exposure → Alerts → Priority → Management Action up
into one executive summary, ending in a human decision handoff.


In [14]:
import executive_decision_center as edc

if not scored_df.empty:
    summary = edc.build_executive_summary(scored_df, alerts_df, actions_df)
    for k, v in summary.items():
        print(f"{k}: {v}")
    print("\nTop financial exposure claims:")
    display(edc.top_financial_exposure_claims(scored_df))
    print("\n" + edc.GOVERNANCE_WARNING)
else:
    print("scored_df is empty — run the earlier cells first.")


total_claims: 1000
total_claim_amount: 573396687.3238914
expected_loss: 220967516.05544814
average_claim: 573396.6873238914
critical_count: 6
high_count: 134
medium_count: 483
low_count: 377
p1_count: 100
p2_count: 172
p3_count: 29
p4_count: 29
total_alerts: 709

Top financial exposure claims:


,Claim_ID,Claim_Type,Customer_Region,Claim_Amount,Risk_Score,Risk_Level
920,CLM00035,Accidental Damage,West,4734075.72,74.41,High
736,CLM00892,Third Party,Central,3535004.49,77.31,Critical
984,CLM00373,Theft,East,3099173.16,65.52,High
815,CLM00852,Third Party,East,3065758.38,66.20,High
509,CLM00617,Third Party,West,3055134.53,73.65,High
699,CLM00151,Theft,West,3015423.50,66.43,High
301,CLM00544,Other,East,2812120.94,65.22,High
2,CLM00683,Accidental Damage,North,2787118.74,68.72,High
268,CLM00383,Theft,West,2744294.43,69.11,High
591,CLM00833,Natural Disaster,Central,2664376.09,65.84,High



GOVERNANCE NOTICE: Insurance Claim Intelligence is an AI-assisted decision-support system. It does NOT automatically approve, reject, investigate, or settle insurance claims. Every risk score, alert, and recommended action on this screen is advisory. Final claim decisions remain the responsibility of authorized human personnel, in accordance with organizational policy and regulatory requirements.


## 18. Routing, Notification & Escalation Workflow

The final application supports department routing across Motor, Property, Health, Travel, Life, Marine, and General Claims.

Routing can be filtered by Department, Priority, and Risk Level, with Claim ID search and claim-level detail.

**Admin Approval is routing approval only — it is not claim approval.**

After routing approval, the application exposes an email section containing editable:

- To
- CC
- Subject
- Body

The message opens through the user's default email application using a standard `mailto:` workflow. The human reviewer checks the message and manually sends it.

The final version does **not** require SMTP, API-based automatic email delivery, or hard-coded recipient addresses.


In [15]:
import notification_engine as ne

if not actions_df.empty:
    workflow = ne.build_escalation_workflow_summary(actions_df)
    display(workflow)

    p1_queue = ne.build_notification_queue(actions_df, "P1")
    if not p1_queue.empty:
        preview = ne.render_email_preview(p1_queue.iloc[0], scored_df)
        print("\nSample P1 notification preview:")
        print("Subject:", preview["subject"])
        print(preview["body"])
else:
    print("actions_df is empty — run the Management Actions cell first.")


,Priority,Label,Queued_Claims,Status
0,P1,Immediate,100,Ready for Email
1,P2,High,172,Ready for Email
2,P3,Monitor,29,Monitoring Queue
3,P4,Review,29,Standard Queue



Sample P1 notification preview:
Subject: Subject: Claim Reporting Details Review
The reporting timeline for your claim requires clarification. Our team will contact you if additional information is required.
Subject: Claim Reporting Details Review

The reporting timeline for your claim requires clarification. Our team will contact you if additional information is required.

SMS Template:
We need to review the incident reporting details before completing your claim assessment. Our team will contact you if clarification is required.
Manager CTA:
REVIEW REPORTING: Check reporting timeline and confirm incident details.

[Source: pipeline-generated communication content, claim CLM00997, priority P1]


## 19. Final Results & Production Validation

### Official Held-Out Model Results

- **Claim Severity:** Random Forest — R² 0.7869, MAE ₹143,067.30, RMSE ₹219,593.00
- **High-Cost Claim:** Random Forest — Accuracy 93.50%, F1 68.29%, ROC-AUC 93.61%

### Final Master Validation

```text
PASS:     21
REVIEW:    3
WARNING:   0
FAIL:      0
```

**Final Status:** `READY TO FREEZE CURRENT MODEL/APPLICATION`

The three REVIEW items are governance/diagnostic observations:
1. False negatives
2. Probability-threshold trade-off
3. Full-data vs documented held-out performance gap

They are not automatic production blockers.

### Current Full-Dataset Diagnostics

**Regression**
- MAE: ₹76,973.14
- RMSE: ₹134,815.22
- R²: 0.9330

**Classification**
- Accuracy: 97.40%
- Precision: 82.46%
- Recall: 94.00%
- F1: 87.85%
- ROC-AUC: 98.97%
- TN: 880 | FP: 20 | FN: 6 | TP: 94

These are **full-dataset diagnostics only**. The documented held-out results remain the official independent performance reference.

### Final Deliverables

- Stable Streamlit application
- Six backend modules
- Two production Random Forest models
- 44-feature validated production schema
- Master production validation suite
- Consolidated validation artifacts
- Final README
- Final 13-slide CLAIMGUARD AI presentation
- Updated complete project notebook


## 20. Business Benefits

- **Faster triage:** high-risk, high-cost, and anomalous claims are
  surfaced automatically instead of relying on manual screening
- **Portfolio-wide visibility:** exposure, expected loss, and risk mix are
  visible in one executive view instead of scattered spreadsheets
- **Consistent escalation:** every claim gets the same P1–P4 logic applied,
  reducing reviewer-to-reviewer inconsistency
- **Audit trail:** each alert and action carries an explicit reason,
  supporting governance and compliance review
- **Human-in-the-loop by design:** the system accelerates review without
  removing human judgment from claim decisions


## 21. Limitations

- The held-out regression and classification metrics retain meaningful prediction error; the system is a screening and prioritization aid, not a source of truth.
- The 0–100 risk-score weights are predefined operational weights rather than continuously learned weights.
- Full-dataset diagnostics may include observations seen during model fitting and therefore are not independent validation.
- Feature importance describes fitted-model reliance and does not establish causality.
- Severe / Critical cases showed larger regression errors in current diagnostic analysis and deserve additional human attention.
- False negatives remain an important review concern for high-cost classification.
- Expected-loss logic is decision-support analytics, not actuarial reserving.
- The final email workflow prepares a message for manual human sending rather than operating as an automated messaging infrastructure.
- Enterprise deployment would require appropriate security, access control, privacy, audit logging, and ongoing model monitoring.


## 22. Future Scope

Potential future enhancements include:

- Role-based access control
- Production audit logging
- Model drift monitoring
- Controlled retraining pipelines
- Calibrated threshold governance
- External validation on newly collected claims
- Additional claim-level explainability
- Integration with an authorized claims-management system
- Enterprise notification integration
- Stronger automated data-quality monitoring

These are future enhancements and are not required for the current validated version.


## 23. Deployment Instructions

### Run Locally

```cmd
cd /d C:\Insurance_Claim_Intelligence
pip install -r requirements.txt
python -m streamlit run app.py
```

Using `python -m streamlit` is recommended where direct execution of `streamlit.exe` is restricted.

### Run Final Master Validation

```cmd
cd /d C:\Insurance_Claim_Intelligence
python claimguard_master_validation.py
```

Validation outputs are stored in:

```text
claimguard_master_validation_results/
```

The master suite is read-only: it does not retrain models, replace `.pkl` files, change thresholds, or modify `app.py`.


## 24. Interview Summary

**What is CLAIMGUARD AI?**  
An AI-assisted insurance claims decision-support platform that combines claim-severity prediction, high-cost risk, operational risk scoring, expected-loss analysis, alerts, management actions, department routing, portfolio intelligence, and human-controlled escalation.

**What ML is used?**  
Two production Random Forest models: a regressor for claim severity and a classifier for high-cost claim risk.

**What is the official performance?**  
Severity: R² 0.7869, MAE ₹143,067.30, RMSE ₹219,593.00.  
High-cost classification: Accuracy 93.50%, F1 68.29%, ROC-AUC 93.61%.

**How many features do the production models use?**  
Both validated production models expect **44 features**. The six risk-engine signals are an operational decision-support subset, not the full model schema.

**What did final validation show?**  
21 PASS, 3 REVIEW, 0 WARNING, 0 FAIL — final status: **READY TO FREEZE CURRENT MODEL/APPLICATION**.

**Why are full-dataset scores higher?**  
They are diagnostics from the fitted models on the available processed dataset, not an independent held-out test. Therefore the documented held-out metrics remain the official performance reference.

**How is governance enforced?**  
CLAIMGUARD AI never automatically approves, rejects, investigates, or settles a claim. Final decisions remain human-authorized.

**What was the feature-selection decision?**  
Reduced-feature experiments did not outperform the production models under the evaluated conditions, so the validated 44-feature production schema and current models were retained.

---

## Final Project Status

**Model Development:** Complete  
**Feature-Selection Evaluation:** Complete  
**Production Schema Validation:** Complete  
**Explainability Analysis:** Complete  
**Robustness & Error Analysis:** Complete  
**Master Validation:** Complete  
**Application:** Stable / Frozen  
**Production Warnings:** 0  
**Production Failures:** 0  

### CLAIMGUARD AI — Final Updated Version

**Nishanth S**
